In [1]:
# %pip install -q "bdp-model-gate" pandas numpy scikit-learn

In [2]:

import json
import logging
import warnings

import numpy as np
import pandas as pd

import bdp_model_gate

# The library logs through the stdlib `logging` module and never calls
# basicConfig() itself, so it composes with whatever your pipeline uses.
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(name)s: %(message)s")
logging.getLogger("bdp_model_gate").setLevel(logging.INFO)
pd.set_option("display.width", 130)

print("bdp-model-gate", bdp_model_gate.__version__)

bdp-model-gate 0.6.0


In [3]:
rng = np.random.default_rng(42)
N = 1500

region = rng.choice(["Lagos", "Abuja", "Kano", "Port Harcourt"], N, p=[
                    0.4, 0.25, 0.2, 0.15])
gender = rng.choice(["F", "M"], N, p=[0.45, 0.55])

income = rng.lognormal(11.6, 0.45, N) * pd.Series(region).map(
    {"Lagos": 1.35, "Abuja": 1.20, "Port Harcourt": 1.00, "Kano": 0.70}
).to_numpy()

X = pd.DataFrame(
    {
        "monthly_income_ngn": income.round(2),
        "age": rng.integers(21, 65, N).astype(float),
        "months_employed": np.clip(rng.normal(48, 30, N), 0, None).round(),
        "existing_loans": rng.poisson(1.1, N).astype(float),
        "debt_to_income": np.clip(rng.beta(2, 5, N) * 1.4, 0.01, 0.95).round(4),
        "distance_to_branch_km": (
            pd.Series(region).map(
                {"Lagos": 2.0, "Abuja": 3.5, "Port Harcourt": 6.0, "Kano": 14.0}
            ).to_numpy()
            + rng.normal(0, 1.1, N)
        ).round(2),
    }
)

logit = (
    -1.0
    + 3.00 * np.log(X["monthly_income_ngn"] / 50_000)
    - 6.00 * X["debt_to_income"]
    + 0.030 * X["months_employed"]
    - 0.50 * X["existing_loans"]
    + 0.90 * (gender == "M")          # nudges the ground truth
)
repaid = rng.binomial(1, 1 / (1 + np.exp(-logit)))
protected_df = pd.DataFrame({"gender": gender, "region": region})

print(f"{N} applicants | repayment rate {repaid.mean():.1%}")
display(X.head())

1500 applicants | repayment rate 61.4%


,monthly_income_ngn,age,months_employed,existing_loans,debt_to_income,distance_to_branch_km
0,122847.44,39.0,21.0,2.0,0.0727,13.06
1,93197.19,59.0,27.0,0.0,0.4136,3.31
2,31799.64,51.0,0.0,5.0,0.3211,5.30
3,34481.98,27.0,15.0,1.0,0.8258,14.43
4,279242.74,25.0,78.0,0.0,0.8625,3.53


In [4]:
repaid[:5]

array([0, 1, 0, 0, 1])

In [5]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val, prot_train, prot_val = train_test_split(
    X, repaid, protected_df, test_size=0.35, random_state=42, stratify=repaid
)

# gender and region are excluded — "fairness through unawareness".
# Section 5 shows why that alone is not enough.
model = GradientBoostingClassifier(random_state=42, n_estimators=120, max_depth=3)
model.fit(X_train, y_train)

# For binary classification y_pred is normally the positive-class probability.
y_pred = model.predict_proba(X_val)[:, 1]

print(f"train {len(X_train)} | validation {len(X_val)}")

train 975 | validation 525


In [6]:
protected_df

,gender,region
0,M,Kano
1,M,Abuja
2,F,Port Harcourt
3,F,Kano
4,M,Lagos
...,...,...
1495,M,Abuja
1496,M,Port Harcourt
1497,F,Abuja
1498,M,Lagos


**StructuredGateContext bundles everything the checks might need. Only the model (or a predict_fn), X, y_true and y_pred are required — every other field is optional, and omitting one makes the checks that depend on it report NOT_APPLICABLE rather than fail**

In [7]:
#  set task explicitly for anything you gate on (binary, multiclass, regression).

from bdp_model_gate import ModelGate, StructuredGateContext

minimal = StructuredGateContext(
    model=model,
    X=X_val,
    y_true=y_val,
    y_pred=y_pred,
    task="auto") # I set this to "auto" intentional, the StructuredGate look into the y_true, if the values 0/1 ==> Binary
minimal_report = ModelGate().run(minimal)
print(minimal_report.summary())


minimal = StructuredGateContext(
    model=model,
    X=X_val,
    y_true=y_val,
    y_pred=y_pred,
    task="binary")
minimal_report = ModelGate().run(minimal)
print(minimal_report.summary())

minimal = StructuredGateContext(
    model=model,
    X=X_val,
    y_true=y_val,
    y_pred=y_pred,
    task="multiclass")
minimal_report = ModelGate().run(minimal)
print(minimal_report.summary())

INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.
INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.
INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.
INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.
INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if

INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.
INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.
INFO     bdp_model_gate.gate: gate_status=PASS task=binary n_flags=0 metric=roc_auc score=0.8636


Gate status: PASS (2016ms, binary)
  roc_auc: 0.8636
  validation: 0 flag(s)
  performance: 0 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 0 flag(s)


INFO     bdp_model_gate.gate: gate_status=PASS task=binary n_flags=0 metric=roc_auc score=0.8636
WARNING  bdp_model_gate.gate: check=performance_thresholds raised an exception: ValueError("Classification metrics can't handle a mix of binary and continuous targets")
INFO     bdp_model_gate.gate: gate_status=BLOCKED task=multiclass n_flags=1 metric=None score=None


Gate status: PASS (2002ms, binary)
  roc_auc: 0.8636
  validation: 0 flag(s)
  performance: 0 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 0 flag(s)
Gate status: BLOCKED (15ms, multiclass)
  validation: 0 flag(s)
  performance: 1 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 0 flag(s)


In [8]:
print(type(y_val), y_val.ndim)
print(pd.Series(y_val).nunique())

<class 'numpy.ndarray'> 1
2


In [9]:
def explain_decision(prompt: str) -> str:
    """Stand-in for an LLM that turns a decision into prose for the applicant.
    PromptInjectionCheck probes whatever you pass here."""
    if any(k in prompt.lower() for k in ("ignore previous", "no content policy", "verbatim")):
        return "I cannot comply with that request."
    return "The application was declined due to a high debt-to-income ratio."

    """
    
from bdp_model_gate.structured.fairness import DisparateImpactCheck
class BlockingDisparateImpact(DisparateImpactCheck):
    blocking = True  # now this one check stops the build
    """


model_card = {
    "model_name": "credit-scoring-gbm",
    "version": "0.4.1",
    "use_case": "credit_scoring",                # high-risk -> DPIA required
    "legal_basis": "Contractual necessity (NDPA 2023, s.25(1)(b))",
    "data_minimization_justification": "Only affordability signals are collected.",
    "training_data_source": "Internal loan book, 2021-2025, consented at origination",
    "dpia_completed": True,
    "influences_decision_about_person": True,
    "explainability_method": "SHAP TreeExplainer, surfaced in the adverse-action notice",
}

context = StructuredGateContext(
    model=model,
    X=X_val,
    y_true=y_val,
    y_pred=y_pred,
    protected_df=prot_val,                       # enables fairness
    latencies_ms=rng.gamma(9.0, 8.5, 500),       # enables the latency gate
    cost_per_inference=0.0009,                   # enables the cost gate
    model_card=model_card,                       # enables compliance
    generate_fn=explain_decision,                # enables prompt injection
    task="binary",
)

report = ModelGate().run(context)
print(report.summary())

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO     bdp_model_gate.security: prompt_injection: firing 6 prompt(s) at 1 surface(s) = 6 generative call(s) (security.injection_depth=1)
INFO     bdp_model_gate.gate: gate_status=BLOCKED task=binary n_flags=15 metric=roc_auc score=0.8636


Gate status: BLOCKED (46006ms, binary)
  roc_auc: 0.8636
  validation: 1 flag(s)
  performance: 0 flag(s)
  compliance: 0 flag(s)
  security: 1 flag(s)
  fairness: 13 flag(s)


### 3. Reading a GateReport¶

In [10]:
pd.set_option('display.max_colwidth', None)

check_report = pd.DataFrame(
        [
            {
                "category": r.category,
                "check": r.check_name,
                "flag": r.flag,
                "blocking": r.blocking,
                "ms": r.duration_ms,
                "detail": (r.detail[:62] + "...") if len(r.detail) > 62 else r.detail,
            }
            for r in report.results
        ]
    )

check_report

,category,check,flag,blocking,ms,detail
0,validation,target_leakage,OK,True,5.52,no single feature approaches the model's own predictive power ...
1,validation,split_overlap,OK,True,0.50,"0 duplicate row(s) within the validation set (0.0%, max 5%) — ..."
2,validation,split_overlap,NOT_APPLICABLE,True,0.50,no X_train supplied — supply the training frame and this check...
3,validation,validation_strategy,VALIDATION_STRATEGY_RISK,True,0.01,model_card.validation_strategy missing — state how the holdout...
4,validation,feature_contract,OK,True,0.02,X matches the 6 feature(s) recorded in model.feature_names_in_...
5,validation,feature_drift,NOT_APPLICABLE,False,0.00,no X_train supplied — train-serve skew needs the training fram...
6,fairness,proxy_correlation,PROXY_RISK,False,3652.67,"distance_to_branch_km correlates with region (eta^2=0.940, q=0..."
7,fairness,disparate_impact,OK,False,210.81,"gender: demographic parity diff=0.001 [0.001, 0.090] at 95% (m..."
8,fairness,disparate_impact,DISPARITY_RISK,False,210.81,"region: demographic parity diff=0.365 [0.254, 0.488] at 95% (m..."
9,fairness,shap_subgroup_gap,UNCERTAIN,False,6445.93,monthly_income_ngn SHAP contribution gap across gender=0.116 —...


In [11]:
print(check_report.iloc[3, check_report.columns.get_loc("detail")])

model_card.validation_strategy missing — state how the holdout...


In [12]:
flags = report.flags
print(f"verdict      : {report.gate_status}")
print(f"task         : {report.task}")
print(f"headline     : {report.model_metric} = {report.model_score}")
print(f"blocking     : {sum(r.blocking for r in flags)}")
print(f"non-blocking : {sum(not r.blocking for r in flags)}")
print(f"wall clock   : {report.total_duration_ms} ms\n")
for r in flags:
    print(f"[{'BLOCK ' if r.blocking else 'review'}] {r.category:11} {r.check_name}")
    print(f"          {r.detail}")

verdict      : BLOCKED
task         : binary
headline     : roc_auc = 0.8636
blocking     : 1
non-blocking : 14
wall clock   : 46006.27 ms

[BLOCK ] validation  validation_strategy
          model_card.validation_strategy missing — state how the holdout was produced, one of: out_of_time, temporal, forward_chaining, walk_forward, random_split, stratified_split, cross_validation, grouped_split
[review] fairness    proxy_correlation
          distance_to_branch_km correlates with region (eta^2=0.940, q=0.006 across 12 comparison(s))
[review] fairness    disparate_impact
          region: demographic parity diff=0.365 [0.254, 0.488] at 95% (max 0.1) — the whole 95% interval sits above 0.100
[review] fairness    shap_subgroup_gap
          monthly_income_ngn SHAP contribution gap across gender=0.116 — 0.215 [0.009, 0.696] at 95% of the mean absolute contribution 0.538 (max 50%) — the estimate stays the right side of 0.500 but the interval reaches 0.696, so this sample cannot rule out a brea

Save Gated Report ==> html / json file

In [13]:
payload = report.to_dict()
print(json.dumps({k: v for k, v in payload.items()
      if k != "results_by_category"}, indent=2))
report.to_html("gate_report.html")
print("\nwrote gate_report.json")

{
  "gate_status": "BLOCKED",
  "task": "binary",
  "model_metric": "roc_auc",
  "model_score": 0.8636,
  "model_auc": 0.8636,
  "n_flags": 15,
  "total_duration_ms": 46006.27
}



wrote gate_report.json


**Gate Evaluation**

In [14]:
from bdp_model_gate import PerformanceConfig
from bdp_model_gate.metrics import BUILTIN_METRICS
from bdp_model_gate.structured.performance import PerformanceThresholdCheck

rows = []
for name, spec in sorted(BUILTIN_METRICS.items()):
    if "binary" not in spec.tasks:
        continue
    result = PerformanceThresholdCheck(
        PerformanceConfig(metric=name, min_score=0.0)
    ).run(context)[0]
    rows.append({"metric": name, "value": result.metadata["value"],
                 "expects": "labels" if spec.needs_hard_labels else "scores"})
display(pd.DataFrame(rows).set_index(
    "metric").sort_values("value", ascending=False))

,value,expects
metric,,
average_precision,0.9067,scores
recall,0.8665,labels
roc_auc,0.8636,scores
f1,0.8404,labels
precision,0.8158,labels
accuracy,0.7981,labels
balanced_accuracy,0.7781,labels


In [15]:
sweep = []
for threshold in (0.3, 0.4, 0.5, 0.6, 0.7):
    row = {"decision_threshold": threshold}
    for name in ("precision", "recall", "f1", "roc_auc"):
        cfg = PerformanceConfig(metric=name, min_score=0.0, decision_threshold=threshold)
        row[name] = PerformanceThresholdCheck(cfg).run(context)[0].metadata["value"]
    sweep.append(row)
display(pd.DataFrame(sweep).set_index("decision_threshold").round(4))
print("roc_auc is flat — it ranks, so the threshold is irrelevant to it.")

,precision,recall,f1,roc_auc
decision_threshold,,,,
0.3,0.7519,0.9410,0.8359,0.8636
0.4,0.7853,0.8975,0.8377,0.8636
0.5,0.8158,0.8665,0.8404,0.8636
0.6,0.8297,0.8168,0.8232,0.8636
0.7,0.8542,0.7640,0.8066,0.8636


roc_auc is flat — it ranks, so the threshold is irrelevant to it.


In [16]:
from sklearn.metrics import fbeta_score


def f2_at_30pct(y_true, y_pred):
    return fbeta_score(y_true, (np.asarray(y_pred) >= 0.30).astype(int), beta=2)


result = PerformanceThresholdCheck(
    PerformanceConfig(metric=f2_at_30pct, min_score=0.85)
).run(context)[0]
print(f"{result.flag:18} {result.detail}")
print("reported as:", result.metadata["metric"])   # from the function __name__

OK                 f2_at_30pct=0.8959 [0.874, 0.918] (min 0.85) — the whole interval sits above 0.850
reported as: f2_at_30pct


In [17]:
from unittest import mock

import bdp_model_gate.metrics as metrics_module
from bdp_model_gate.exceptions import GateConfigurationError

# Simulate a core-only install (no scikit-learn) without uninstalling.
with mock.patch.object(metrics_module, "_load_sklearn_metric", return_value=None):
    auto = PerformanceThresholdCheck(
        PerformanceConfig(metric="auto", min_score=0.80)
    ).run(context)[0]
    print("auto fell back to :", auto.metadata["metric"])
    print("flagged as fallback:", auto.metadata["metric_is_fallback"])
    print("detail             :", auto.detail)

    # An explicitly named metric is never swapped for another.
    try:
        PerformanceThresholdCheck(PerformanceConfig(metric="roc_auc")).run(context)
    except GateConfigurationError as exc:
        print("\nexplicit metric unavailable ->", exc)

WARNING  bdp_model_gate.metrics: performance.metric='auto': 'roc_auc' is unavailable (scikit-learn not installed) — scoring with 'accuracy' instead. Set performance.metric explicitly to silence this, and remember min_score is interpreted against 'accuracy', not 'roc_auc'.


auto fell back to : accuracy
flagged as fallback: True
detail             : accuracy=0.7981 [0.762, 0.832] (min 0.8) [fell back from the preferred metric — scikit-learn not installed; computed without scikit-learn] — the estimate breaches 0.800 but the interval reaches back to 0.832, so this sample cannot confirm a breach

explicit metric unavailable -> performance.metric='roc_auc' requires scikit-learn — install it with `pip install bdp-model-gate[structured]`, or set performance.metric to one of: accuracy, lorenz_gini, mae, mape, poisson_deviance, r2, rmse


In [18]:
for r in report.by_category("fairness"):
    print(f"{'  ' if r.is_ok else '->'} {r.check_name:22} {r.flag}")
    print(f"     {r.detail}")

-> proxy_correlation      PROXY_RISK
     distance_to_branch_km correlates with region (eta^2=0.940, q=0.006 across 12 comparison(s))
   disparate_impact       OK
     gender: demographic parity diff=0.001 [0.001, 0.090] at 95% (max 0.1) — the whole interval sits below 0.100
-> disparate_impact       DISPARITY_RISK
     region: demographic parity diff=0.365 [0.254, 0.488] at 95% (max 0.1) — the whole 95% interval sits above 0.100
-> shap_subgroup_gap      UNCERTAIN
     monthly_income_ngn SHAP contribution gap across gender=0.116 — 0.215 [0.009, 0.696] at 95% of the mean absolute contribution 0.538 (max 50%) — the estimate stays the right side of 0.500 but the interval reaches 0.696, so this sample cannot rule out a breach
-> shap_subgroup_gap      UNCERTAIN
     debt_to_income SHAP contribution gap across gender=0.111 — 0.206 [0.011, 0.547] at 95% of the mean absolute contribution 0.538 (max 50%) — the estimate stays the right side of 0.500 but the interval reaches 0.547, so this samp

In [19]:
from bdp_model_gate.structured.fairness import DisparateImpactCheck, ProxyCorrelationCheck

for r in ProxyCorrelationCheck().run(context):
    print(f"{r.flag:14} {r.detail}")

display(
    X_val.assign(region=prot_val["region"].values)
    .groupby("region")[["distance_to_branch_km", "monthly_income_ngn"]]
    .mean()
    .round(2)
)

PROXY_RISK     distance_to_branch_km correlates with region (eta^2=0.940, q=0.006 across 12 comparison(s))


,distance_to_branch_km,monthly_income_ngn
region,,
Abuja,3.39,139551.44
Kano,14.20,84878.88
Lagos,2.10,167986.63
Port Harcourt,6.00,121876.68


In [20]:
for r in DisparateImpactCheck().run(context):
    print(f"{r.flag:18} {r.detail}")

OK                 gender: demographic parity diff=0.001 [0.001, 0.090] at 95% (max 0.1) — the whole interval sits below 0.100
DISPARITY_RISK     region: demographic parity diff=0.365 [0.254, 0.488] at 95% (max 0.1) — the whole 95% interval sits above 0.100


In [21]:
from bdp_model_gate import FairnessConfig
from bdp_model_gate.structured.calibration_checks import (
    CalibrationCheck,
    EqualisedOddsCheck,
    SubgroupCalibrationCheck,
)

FAMILY = {
    "disparate_impact": "independence",
    "equalised_odds": "separation",
    "subgroup_calibration": "sufficiency",
}

for check in (DisparateImpactCheck(), EqualisedOddsCheck(), SubgroupCalibrationCheck()):
    for r in check.run(context):
        if r.metadata.get("protected_attr") != "region":
            continue
        print(f"{FAMILY[r.check_name]:14} {r.flag:26} {r.detail[:74]}")

independence   DISPARITY_RISK             region: demographic parity diff=0.365 [0.254, 0.488] at 95% (max 0.1) — th
separation     UNCERTAIN                  region: true-positive-rate difference=0.219 [0.095, 0.385] at 95% (max 0.1
separation     EQUALISED_ODDS_RISK        region: equalised odds difference=0.263 [0.202, 0.445] at 95% (max 0.1) — 
sufficiency    UNCERTAIN                  region: calibration error spans 0.0500 (Lagos) to 0.1165 (Abuja) — gap 0.0
